In [1]:
import joblib
import pandas as pd
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
import warnings

In [2]:
target_column = "health_condition"

In [3]:
X_train = pd.read_csv("../data/intermediate/train_features.csv")
X_valid = pd.read_csv("../data/intermediate/valid_features.csv")
X_test = pd.read_csv("../data/intermediate/test_features.csv")

y_train = pd.read_csv("../data/intermediate/train_labels.csv")
y_valid = pd.read_csv("../data/intermediate/valid_labels.csv")

X_train['diet_type'] = X_train['diet_type'].astype('category')
X_valid['diet_type'] = X_valid['diet_type'].astype('category')
X_test['diet_type'] = X_test['diet_type'].astype('category')

X_train['gender'] = X_train['gender'].astype('category')
X_valid['gender'] = X_valid['gender'].astype('category')
X_test['gender'] = X_test['gender'].astype('category')

X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 552070 entries, 0 to 552069
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   sleep_duration                491266 non-null  float64 
 1   heart_rate                    545802 non-null  float64 
 2   bmi                           541053 non-null  float64 
 3   calorie_expenditure           509705 non-null  float64 
 4   step_count                    540909 non-null  float64 
 5   exercise_duration             546550 non-null  float64 
 6   water_intake                  517224 non-null  float64 
 7   diet_type                     546580 non-null  category
 8   stress_level                  485727 non-null  float64 
 9   sleep_quality                 505469 non-null  float64 
 10  physical_activity_level       522744 non-null  float64 
 11  smoking_alcohol               529184 non-null  float64 
 12  gender                        535022 non-

In [4]:
X = pd.concat([X_train, X_valid], axis=0)
y = pd.concat([y_train, y_valid], axis=0)

In [5]:
df_submission = pd.read_csv("../data/sample_submission.csv")
df_submission.info()

<class 'pandas.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                295753 non-null  int64
 1   health_condition  295753 non-null  str  
dtypes: int64(1), str(1)
memory usage: 4.5 MB


In [6]:
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
valid_sample_weight = compute_sample_weight(class_weight="balanced", y=y_valid)
y_sample_weight = compute_sample_weight(class_weight="balanced", y=y)

model = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=8, class_weight='balanced')
model.fit(X_train, y_train, sample_weight=train_sample_weight, X_val=X_valid, y_val=y_valid, sample_weight_val=valid_sample_weight)

joblib.dump(model, '../models/hgbc_80.pkl')

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['../models/hgbc_80.pkl']

In [ ]:
from sklearn.metrics import balanced_accuracy_score

y_pred = model.predict(X_valid)

val_score = balanced_accuracy_score(y_valid, y_pred, sample_weight=valid_sample_weight)
print("Validation Balanced Accuracy:", val_score)

In [7]:
model = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=8, class_weight='balanced')
model.fit(X, y, sample_weight=y_sample_weight)

joblib.dump(model, '../models/hgbc_100.pkl')

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['../models/hgbc_100.pkl']

In [8]:
y_pred = model.predict(X_test)

df_submission[target_column] = y_pred
df_submission[target_column] = df_submission[target_column].replace({0:'unhealthy', 1:'at-risk', 2: 'fit'})

df_submission.to_csv('../results/hgbc_baseline.csv', index=False)
df_submission

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
...,...,...
295748,985836,fit
295749,985837,at-risk
295750,985838,unhealthy
295751,985839,fit
